## RBAC — bindings, subjects & `auth can-i`

A `Role` is *just permissions* — it applies to no one until you **bind** it.

- **`RoleBinding`** — namespace-scoped; binds a Role *or* a ClusterRole to subjects in one namespace.
- **`ClusterRoleBinding`** — cluster-scoped; binds a ClusterRole across the whole cluster.

```yaml
kind: RoleBinding
metadata: { name: alice-reads, namespace: team-a }
subjects:
  - { kind: User, name: alice, apiGroup: rbac.authorization.k8s.io }
  - { kind: ServiceAccount, name: ci-runner, namespace: ci }
roleRef: { kind: Role, name: pod-reader, apiGroup: rbac.authorization.k8s.io }
```

Subjects are `User`, `ServiceAccount`, or `Group`. **The crossover trick:** a `RoleBinding` can bind a **ClusterRole** — then its permissions apply *only in that namespace*. Define one `pod-reader` ClusterRole, bind it per namespace — the standard "reuse across namespaces" pattern.

**Verify with `kubectl auth can-i`** — the most useful RBAC command:

```bash
kubectl auth can-i list pods -n team-a
kubectl auth can-i create deployments --as alice
kubectl auth can-i '*' '*' --all-namespaces      # am I cluster-admin?
```

**Ship-with defaults** — four `ClusterRole`s to know: **`cluster-admin`** (everything), **`admin`** (full namespace, not the namespace/quota itself), **`edit`** (read-write workloads, no RBAC), **`view`** (read-only, **excludes Secrets** deliberately). They're **aggregated** — add a ClusterRole labelled `rbac.authorization.k8s.io/aggregate-to-view: "true"` and it folds into `view`. Don't bind directly to `system:*` roles. Imperative: `kubectl create role/rolebinding ... --dry-run=client -o yaml`. On our map these are the **RoleBinding**, **ClusterRoleBinding**, and **subject** chips — the wires that turn permissions into access, gating the arrow to the api-server.